## 실습 개요

실습 목표를 위해 아래 두가지 학습이 필요합니다
#### Supervised Fine-Tuning (SFT)
- `모델에게 정답을 강제로 보여주면서 학습`

#### Direct Preference Optimization (DPO)
- `사용자 선호도를 반영한 pairwise 학습으로 인간처럼 더 선호되는 출력을 생성`

이 노트북에서는 준비된 instruction 데이터셋과 선호도 데이터셋을 바탕으로 두 가지 파인튜닝 전략을 수행합니다.

## Supervised Fine-Tuning (SFT)
### 라이브러리 Import

In [ ]:
import os

# 미리 저장된 모델을 사용하기 위해 환경변수 설정
os.environ["HF_HUB_CACHE"] = "/mnt/elice/dataset/hub"

In [ ]:
import torch  # 딥러닝 프레임워크

# 모델 및 토크나이저 불러오기 위한 huggingface transformers
from transformers import (
    AutoModelForCausalLM,  # 사전 학습된 언어 모델 로드용
    AutoTokenizer,  # 텍스트를 토큰으로 변환하는 토크나이저
    BitsAndBytesConfig,  # 4bit 양자화 설정을 위한 구성 클래스
)

# 학습 효율화를 위한 PEFT (Parameter-Efficient Fine-Tuning) 모듈
from peft import (
    prepare_model_for_kbit_training,  # 양자화된 모델을 LoRA 학습 가능하도록 변환
    LoraConfig,  # LoRA 설정 구성
    get_peft_model,  # LoRA 구성 적용
    AutoPeftModelForCausalLM,  # 학습된 LoRA 모델 불러오기
)

# 데이터셋 로딩 및 변환용 라이브러리
from datasets import Dataset  # pandas -> huggingface Dataset 변환용
import pandas as pd  # CSV 파일 로딩 등 데이터 처리

# 최대 길이 설정
MAX_SEQ_LEN = 1024

#### 1. 데이터 불러오기

In [ ]:
# 사전 포맷팅된 instruction 기반 학습 데이터셋 불러오기
# text 컬럼은 Llama instruct prompt 형식을 따름
df = pd.read_csv("data/processed/kr_finetuning_dataset.csv")

# huggingface Dataset으로 변환 (SFTTrainer가 이 형식을 요구함)
dataset = Dataset.from_pandas(df[["text"]])

# 데이터 확인
print(type(dataset))  # 데이터셋 정보
print("---------------------------------------------------------------------")
print(dataset)
print("---------------------------------------------------------------------")
print(dataset[0]["text"])  # text 컬럼만 존재하며, 답변이 단답으로 되어있음

#### 2. 토크나이저 로드 및 설정

In [ ]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# 해당 모델에 맞는 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 패딩 토큰 수동 설정 (Llama는 기본 pad token이 없음)
tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
tokenizer.pad_token = "<|pad|>"
tokenizer.padding_side = "right"  # 오른쪽 패딩으로 설정

##### 2.1 토큰화 결과 기준 데이터셋 길이 필터링

In [ ]:
def filter_long_sequences(dataset, tokenizer, max_length=1024):
    """max_length를 초과하는 시퀀스를 필터링"""
    def is_valid_length(example):
        tokens = tokenizer.encode(example["text"])
        return len(tokens) <= max_length
    
    return dataset.filter(is_valid_length)

In [ ]:
# 데이터셋 필터링 적용
dataset = filter_long_sequences(dataset, tokenizer, max_length=MAX_SEQ_LEN)

#### 3. 양자화 모델 로드 및 PEFT 준비

In [ ]:
# 사전 학습 모델 로드 (4bit 양자화)
# 4bit 양자화를 위한 설정 객체
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16  # 4bit 로드 여부  # 연산 dtype 설정 (속도 + 정밀도 균형)
)

# 양자화 설정 적용하여 Llama 모델 로드
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,  # 위에서 설정한 4bit 양자화 config 사용
    device_map="auto",  # 사용 가능한 GPU 자동 할당
)

# 모델을 PEFT 학습 가능 상태로 변환 (gradient checkpoint 해제 등)
base_model = prepare_model_for_kbit_training(base_model)

# 토크나이저 크기에 맞게 모델의 임베딩 레이어 리사이즈
base_model.resize_token_embeddings(len(tokenizer))
base_model.config.use_cache = False

In [ ]:
from transformers import TextStreamer

# 모델명 및 양자화 설정
model_name = "meta-llama/Llama-3.2-1B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

# 토크나이저 및 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
tokenizer.pad_token = "<|pad|>"
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

# 추론을 위한 스트리머 (선택)
streamer = TextStreamer(tokenizer)


# 🔧 프롬프트 템플릿 함수
def build_prompt(user_question: str) -> str:
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful and honest assistant.<|start_header_id|>user<|end_header_id|>
{user_question}<|start_header_id|>assistant<|end_header_id|>
"""


# 🔎 사용자 입력
question = input("질문을 입력하세요: ")  # 예: "파이썬에서 리스트를 뒤집는 방법을 설명해줘."

# 프롬프트 구성
prompt = build_prompt(question)

# 토크나이즈
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 응답 생성
with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=300, temperature=0.7, top_p=0.9, streamer=streamer  # 실시간 출력 (선택)
    )

#### 4. LoRA 설정 및 모델 준비

In [ ]:
# LoRA를 적용할 대상 모듈들과 관련 파라미터 설정
peft_config = LoraConfig(
    r=16,  # 랭크: 저차원 공간 크기
    lora_alpha=16,  # 스케일링 계수
    lora_dropout=0.1,  # 드롭아웃 확률
    task_type="CAUSAL_LM",  # causal language modeling task
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # LoRA를 적용할 linear layer 이름 목록
)

# 위 설정을 기반으로 LoRA가 적용된 PEFT 모델 생성
base_model = get_peft_model(base_model, peft_config)

#### 5. 데이터 포맷 함수 정의

In [ ]:
# SFTTrainer에 넘길 데이터셋을 어떻게 문자열로 변환할지 정의
# 여기서는 이미 완성된 텍스트를 그대로 반환
# (Instruction Prompt 형식이 text 컬럼에 완성되어 있으므로)
def formatting_prompts_func(example):
    return example["text"]

#### 6. 데이터 Collator 준비 (응답 토큰만 학습)

In [ ]:
from trl import DataCollatorForCompletionOnlyLM

# assistant 영역만 loss를 계산하도록 하는 collator
response_template = "<|start_header_id|>assistant<|end_header_id|>"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer
)

#### 7. 학습 설정 및 Trainer 정의

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_args = SFTConfig(
    output_dir="outputs/sft_model",  # 모델 저장 경로
    per_device_train_batch_size=2,  # 배치 크기 (GPU 메모리 맞게 조절)
    gradient_accumulation_steps=1,  # 작은 배치일 때 누적 단계 설정
    num_train_epochs=2,  # 전체 에폭 수
    learning_rate=2e-5,  # 학습률
    bf16=True,  # bfloat16 사용 여부
    logging_steps=10,  # 로그 출력 주기
    save_total_limit=2,  # 저장할 체크포인트 개수 제한
    max_seq_length=MAX_SEQ_LEN,  # 최대 시퀀스 길이
    optim="paged_adamw_32bit",  # 페이지 기반 AdamW 옵티마이저 (양자화 모델 최적화)
)

trainer = SFTTrainer(
    model=base_model,  # LoRA 적용된 모델
    args=sft_args,  # 학습 인자들
    train_dataset=dataset,  # 학습 데이터셋
    formatting_func=formatting_prompts_func,  # 포맷팅 함수
    data_collator=collator,  # Collator (assistant만 학습)
    peft_config=peft_config,  # PEFT 설정
)

### 8. SFT 학습 수행

모델 학습에는 약 1시간 정도 소요됩니다. 학습 결과가 미리 준비되어 있으므로, 이 셀의 실행을 강제로 끊고, 다음 셀로 넘어가셔도 무방합니다.

In [ ]:
# 학습 시작!
trainer.train()

# 학습된 adapter 저장 (base model은 저장 안됨)
trainer.save_model("outputs/sft_model/final_adapter")

# 토크나이저 저장 (나중에 추론 시 필요)
tokenizer.save_pretrained("outputs/sft_model/final_adapter")

### 9. 학습된 LoRA 모델 병합 및 저장

In [ ]:
# CPU로 로드 후 full-weight 모델 병합
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    "outputs/sft_model/final_adapter", device_map="cpu", torch_dtype=torch.bfloat16
)

# LoRA weight를 원본 모델에 병합 (추론 최적화 목적)
merged_model = merged_model.merge_and_unload()

# 저장 디렉토리 설정 및 저장
output_path = "outputs/sft_model/merged"
merged_model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

SFT 학습이 끝났습니다!  
다음 장에서는 학습이 끝난 모델을 DPO 알고리즘으로 추가 학습 시켜보겠습니다.

### 실습 : 학습이 끝난 모델을 로드하여 추론하기  

아래 요구사항을 참고하여, 각 요구사항에 대응하는 코드의 None을 적절한 코드로 대체하여 학습이 끝난 모델을 로드하여 추론하는 코드를 완성하세요. 

이때, GPU 메모리 부족으로 인한 오류를 방지하기 위해, 커널을 재시작한 후 아래 코드를 실행하세요.  

요구사항 1 : 모델 로드는 다음 방식으로 진행하세요:
 - outputs/sft_model/merged 경로에 저장된 병합된 모델을 sft_model로 불러옵니다.    
  
요구사항 2 : 프롬프트는 LLaMA Instruct 포맷을 따르도록 구성하세요:
 - system / user / assistant 헤더를 포함해야 합니다.
 - 사용자 질문은 input() 함수로 직접 입력받도록 구현하세요.

참고: 테스트용 예시 질문 (학습데이터와 비슷한 양식, 정보와 질문을 모두 입력하세요) 

정보:
대구 FC는 2002년 FIFA 월드컵의 축구 붐에 힘입어 대한민국 최초로 시민 구단의 개념을 도입하면서 창단되었다. 월드컵이 끝난 후 2002년 8월 6일 창립 회의를 시작으로 2002년 10월 9일 (주)대구시민프로축구단이라는 공식 명칭으로 첫 창립 총회를 열면서 창단하게 된다. 이후 2002년 11월 15일부터 12월 24일간 1차 시민주 공모를 실시해 127억원의 자금을 확보하였으며, 2002년 12월 26일 한국프로축구연맹에 의해 창단이 승인되었다. 2003년 3월 19일 공식 창단식을 거행하였으며 초대 감독으로 1980년대와 1990년대 대한민국 국가대표팀을 이끌었던 박종환을 선임하고 K리그 2003 시즌에 참가하였다.

질문:
1. 대구 FC는 몇 년에 창단되었나요?
2. 초대 감독은 누구였나요?
3. 첫 시즌에 확보한 자금은 얼마였나요?

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
from peft import PeftModel

# 공통 설정
max_new_tokens = 300

# LLaMA 3 스타일 프롬프트 템플릿 (시작 토큰 포함)
prompt_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{}\n<|eot_id|><|start_header_id|>user<|end_header_id|>
{}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

# Step 1. 사용자 질문 입력 (요구사항 2-1)
user_prompt = None

# Step 2. 프롬프트 생성
system_prompt = "주어진 정보를 바탕으로 질문에 순서대로 답하세요."
full_prompt = prompt_template.format(system_prompt, user_prompt)

# Step 3. tokenizer 로드 (LoRA adapter와 함께 저장된 디렉토리)
merged_model_path = "outputs/sft_model/merged"
tokenizer = AutoTokenizer.from_pretrained(merged_model_path)

# 특수 토큰 등록 (streamer 및 디코딩 시 필수)
tokenizer.add_special_tokens(
    {"additional_special_tokens": ["<|start_header_id|>", "<|end_header_id|>", "<|eot_id|>", "<|begin_of_text|>"]}
)

# Step 4. 병합된 모델 로드 (요구사항 1)
sft_model = None

# Step 5. streamer 설정
streamer = TextStreamer(tokenizer, skip_special_tokens=True, skip_prompt=True)

# Step 6. 추론 및 실시간 출력
inputs = tokenizer(full_prompt, return_tensors="pt").to(sft_model.device)

with torch.no_grad():
    sft_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.5,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        streamer=streamer
    )